In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

class DataPreprocessing:
    def __init__(self, file_path, target_col="PM2.5"):
        self.file_path = file_path
        self.target_col = target_col
        self.df = None
        self.scaler = RobustScaler()

    def loading(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    # REMOVED: This causes Data Leakage (double shift) because dataset.py already shifts for sequences.
    # We leave the code commented out for reference.
    # def create_target_column(self):
    #     self.df[self.target_col] = self.df[self.target_col].shift(-1)
    #     self.df.dropna(inplace=True)
    #     return self.df

    def handle_datetime(self):
        self.df["datetime"] = pd.to_datetime(self.df[["year", "month", "day", "hour"]])
        self.df["day_of_week"] = self.df["datetime"].dt.dayofweek

        return self.df

    def handle_missing_value(self):
        numeric_cols = self.df.select_dtypes(include=["int64", "float64"]).columns
        categorical_cols = self.df.select_dtypes(include=["object"]).columns

        self.df[numeric_cols] = self.df[numeric_cols].interpolate(limit=6)
        self.df[categorical_cols] = self.df[categorical_cols].fillna(method="bfill")

        self.df.dropna(inplace=True)
        return self.df

    def handle_encoding(self):
        self.df = pd.get_dummies(self.df, drop_first=True)
        return self.df

    # REMOVED: Scaling the entire dataset before splitting causes Data Leakage from the test set.
    # Scaling is now handled in dataset.py after the Train/Test split.
    # def handle_scaling(self):
    #     return self.df

    def save_processed(self, output_path):
        self.df.to_csv(output_path, index=False)

    def process(self, output_path):
        self.loading()
        self.handle_datetime()
        self.handle_missing_value()
        # self.create_target_column() # Commented out to prevent Data Leakage (Future Shift)
        self.handle_encoding()
        # self.handle_scaling() # Commented out, moved to dataset.py to prevent Data Leakage
        self.save_processed(output_path)
        return self.df

data = DataPreprocessing(file_path="/PRSA_Aotizhongxin.csv")
processed_data = data.process(output_path="processed_data_v1.csv")


/tmp/ipykernel_8676/599192494.py:34: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.df[categorical_cols] = self.df[categorical_cols].fillna(method="bfill")


In [5]:
processed_data.values

array([[1, 2013, 3, ..., False, False, False],
       [2, 2013, 3, ..., False, False, False],
       [3, 2013, 3, ..., False, False, False],
       ...,
       [35062, 2017, 2, ..., False, False, False],
       [35063, 2017, 2, ..., False, False, False],
       [35064, 2017, 2, ..., False, False, False]], dtype=object)

In [6]:
len(processed_data)

33610

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

class TimeSeriesDataset:
    def __init__(self, file_path, target_col="PM2.5", seq_len=24, test_ratio=0.2):
        self.file_path = file_path
        self.target_col = target_col
        self.seq_len = seq_len
        self.test_ratio = test_ratio

        self.df = None
        self.scaler = RobustScaler() # Added Scaler here

    def load_data(self):
        self.df = pd.read_csv(self.file_path) # processed_data_v1.csv
        self.df.drop("datetime", axis=1, inplace=True)
        return self.df

    def split_data(self):
        split_idx = int(len(self.df) * (1 - self.test_ratio))

        train_df = self.df[:split_idx].reset_index(drop=True)
        test_df = self.df[split_idx:].reset_index(drop=True)

        return train_df, test_df

    def scale_data(self, train_df, test_df):
        # Fit ONLY on training data to prevent Data Leakage
        cols = train_df.columns

        train_scaled = self.scaler.fit_transform(train_df)
        train_df = pd.DataFrame(train_scaled, columns=cols)

        # ONLY transform test (using the rules learned from train)
        test_scaled = self.scaler.transform(test_df)
        test_df = pd.DataFrame(test_scaled, columns=cols)

        return train_df, test_df

    def create_sequences(self, data):
        X, y = [], []

        values = data.values
        target_idx = data.columns.get_loc(self.target_col)

        for i in range(len(data) - self.seq_len):
            X.append(values[i:i + self.seq_len])
            y.append(values[i + self.seq_len, target_idx])

        return np.array(X), np.array(y)

    def get_data(self):
        self.load_data()

        train_df, test_df = self.split_data()

        # Added scaling step here to prevent data leakage
        train_df, test_df = self.scale_data(train_df, test_df)

        X_train, y_train = self.create_sequences(train_df)
        X_test, y_test = self.create_sequences(test_df)

        return X_train, y_train, X_test, y_test

In [3]:
dataset = TimeSeriesDataset(file_path="/content/processed_data_v1.csv", target_col="PM2.5", seq_len=24, test_ratio=0.2)
X_train, y_train, X_test, y_test = dataset.get_data()

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape


((26864, 24, 32), (26864,), (6698, 24, 32), (6698,))

In [5]:
import torch
import torch.nn as nn

class AQIModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(AQIModel, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out




In [6]:
len(X_train)

26864

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# Correctly derive input_size from the shape of X_train
input_size = X_train.shape[2] # This will be 32, matching the actual data dimensions
hidden_size = 100
output_size = 1
num_layers = 2
learning_rate = 0.001
batch_size = 32
epochs = 100
model = AQIModel(input_size, hidden_size, output_size, num_layers)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to the device
model.to(device)


criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Training on {device}")
for epoch in range(epochs):
    for i in range(0, len(X_train), batch_size):
        # Convert numpy arrays to torch tensors and move to device
        inputs = torch.from_numpy(X_train[i:i+batch_size]).float().to(device)
        targets = torch.from_numpy(y_train[i:i+batch_size]).float().to(device).unsqueeze(1) # unsqueeze for (batch_size, 1) target shape

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    # Print loss once per epoch for cleaner output
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

print("Training complete")

torch.save(model.state_dict(), "/model_v1.pth")
print("Model saved to /model.pth")

Training on cuda
Epoch 1, Loss: 0.0078
Epoch 2, Loss: 0.0067
Epoch 3, Loss: 0.0065
Epoch 4, Loss: 0.0065
Epoch 5, Loss: 0.0068
Epoch 6, Loss: 0.0073
Epoch 7, Loss: 0.0072
Epoch 8, Loss: 0.0064
Epoch 9, Loss: 0.0066
Epoch 10, Loss: 0.0073
Epoch 11, Loss: 0.0067
Epoch 12, Loss: 0.0060
Epoch 13, Loss: 0.0066
Epoch 14, Loss: 0.0041
Epoch 15, Loss: 0.0065
Epoch 16, Loss: 0.0058
Epoch 17, Loss: 0.0058
Epoch 18, Loss: 0.0056
Epoch 19, Loss: 0.0057
Epoch 20, Loss: 0.0058
Epoch 21, Loss: 0.0059
Epoch 22, Loss: 0.0073
Epoch 23, Loss: 0.0052
Epoch 24, Loss: 0.0049
Epoch 25, Loss: 0.0061
Epoch 26, Loss: 0.0060
Epoch 27, Loss: 0.0057
Epoch 28, Loss: 0.0059
Epoch 29, Loss: 0.0055
Epoch 30, Loss: 0.0043
Epoch 31, Loss: 0.0044
Epoch 32, Loss: 0.0056
Epoch 33, Loss: 0.0039
Epoch 34, Loss: 0.0055
Epoch 35, Loss: 0.0038
Epoch 36, Loss: 0.0047
Epoch 37, Loss: 0.0038
Epoch 38, Loss: 0.0043
Epoch 39, Loss: 0.0031
Epoch 40, Loss: 0.0037
Epoch 41, Loss: 0.0040
Epoch 42, Loss: 0.0034
Epoch 43, Loss: 0.0050
Epo

In [10]:
print('Loading and preprocessing PRSA_Dongsi.csv...')
dongsi_data_preprocessor = DataPreprocessing(file_path="/PRSA_Dongsi.csv")
processed_dongsi_data = dongsi_data_preprocessor.process(output_path="processed_data_dongsi.csv")
print('PRSA_Dongsi.csv preprocessed and saved to processed_data_dongsi.csv')

Loading and preprocessing PRSA_Dongsi.csv...


/tmp/ipykernel_8676/599192494.py:34: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.df[categorical_cols] = self.df[categorical_cols].fillna(method="bfill")


PRSA_Dongsi.csv preprocessed and saved to processed_data_dongsi.csv


In [11]:
print('Creating time series sequences for processed_data_dongsi.csv...')
dongsi_dataset = TimeSeriesDataset(file_path="/content/processed_data_dongsi.csv", target_col="PM2.5", seq_len=24, test_ratio=0.2) # Use the same seq_len and test_ratio
# We'll get X_train, y_train, X_test, y_test from this dataset, but we'll primarily use the test split for predictions
_, _, X_dongsi_test, y_dongsi_test = dongsi_dataset.get_data()

print(f"Shape of X_dongsi_test: {X_dongsi_test.shape}")
print(f"Shape of y_dongsi_test: {y_dongsi_test.shape}")

Creating time series sequences for processed_data_dongsi.csv...
Shape of X_dongsi_test: (6240, 24, 32)
Shape of y_dongsi_test: (6240,)


In [13]:
print('Loading the trained model and making predictions...')

# Instantiate the model with the same architecture parameters
# Ensure input_size is consistent (it should be 32, as determined before)
model_for_prediction = AQIModel(input_size=X_train.shape[2], hidden_size=hidden_size, output_size=output_size, num_layers=num_layers)
model_for_prediction.to(device)

# Load the saved state dictionary
model_for_prediction.load_state_dict(torch.load("/model_v1.pth", map_location=device))
model_for_prediction.eval() # Set the model to evaluation mode

# Convert new test data to tensor and move to device
inputs_dongsi = torch.from_numpy(X_dongsi_test).float().to(device)

# Make predictions
with torch.no_grad():
    predictions_dongsi = model_for_prediction(inputs_dongsi).cpu().numpy()

print("Predictions made on PRSA_Dongsi.csv!")
#print(f"First 5 predictions: {predictions_dongsi[:10].flatten()}")
#print(f"First 5 actual values (from dongsi_dataset): {y_dongsi_test[:10]}")

for i in range(5):
    print(f"Prediction {i+1}: {predictions_dongsi[i]}, Actual: {y_dongsi_test[i]}")

Loading the trained model and making predictions...
Predictions made on PRSA_Dongsi.csv!
Prediction 1: [0.47599566], Actual: 0.46808510638297873
Prediction 2: [0.6113087], Actual: 0.5957446808510638
Prediction 3: [0.5980294], Actual: 0.7021276595744681
Prediction 4: [0.90765786], Actual: 0.7872340425531915
Prediction 5: [0.7476126], Actual: 0.723404255319149


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Ensure predictions and actuals are 1D arrays for metric calculation
actuals = y_dongsi_test
predictions = predictions_dongsi.flatten()

mae = mean_absolute_error(actuals, predictions)
mse = mean_squared_error(actuals, predictions)
rmse = np.sqrt(mse)

print(f"\nEvaluation Metrics for PRSA_Dongsi.csv:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

print("\nInterpretation:")
print("MAE represents the average absolute difference between predicted and actual values.")
print("MSE and RMSE penalize larger errors more heavily. Lower values are better.")


Evaluation Metrics for PRSA_Dongsi.csv:
Mean Absolute Error (MAE): 0.1391
Mean Squared Error (MSE): 0.0635
Root Mean Squared Error (RMSE): 0.2521

Interpretation:
MAE represents the average absolute difference between predicted and actual values.
MSE and RMSE penalize larger errors more heavily. Lower values are better.
